### 데이터 조회를 위한 도구 설치 및 API키 연결

In [4]:
import os
import io
import zipfile
import xml.etree.ElementTree as ET
import json

import pandas as pd
import requests
from dotenv import load_dotenv

# 프로젝트 루트의 .env 파일을 읽습니다.
load_dotenv("/Users/anna/bulsae/.env")

# API 키는 변수에만 저장하고 화면에 출력하지 않습니다.
dart_api_key = os.getenv("DART_API_KEY")

if not dart_api_key:
    raise ValueError(".env 파일에 DART_API_KEY가 있는지 확인해 주세요.")

print("API 키를 확인했습니다.")

API 키를 확인했습니다.


In [6]:
# 사전에 설정한 기업 불러오기

file_path = "/Users/anna/bulsae/data/shared/companies.jsonl"

with open(file_path, encoding="utf-8") as file:
    companies = [json.loads(line) for line in file if line.strip()]

companies_df = pd.DataFrame(companies)
display(companies_df)

,company_id,input_name,legal_name,dart_corp_code
0,COMPANY-INNISFREE,이니스프리,주식회사 이니스프리,00812353
1,COMPANY-ESPOIR,에스쁘아,주식회사 에스쁘아,01061150
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,주식회사 아모레퍼시픽,00583424
3,COMPANY-ETUDE,에뛰드,주식회사 에뛰드,00183808
4,COMPANY-PACIFICTECH,퍼시픽테크,주식회사 퍼시픽테크,01784251
5,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,주식회사 아모스프로페셔널,00144456
6,COMPANY-COSRX,코스알엑스,주식회사 코스알엑스,01364491


In [8]:
# 사전에 설정한 기업 목록 한정 Dart API에서 제공하는 데이터 불러오기 
url = "https://opendart.fss.or.kr/api/company.json"

results = []

for company in companies:
    response = requests.get(
        url,
        params={
            "crtfc_key": dart_api_key,
            "corp_code": company["dart_corp_code"],
        },
        timeout=30,
    )

    if response.status_code != 200:
        print(f'{company["input_name"]}: HTTP 오류 {response.status_code}')
        continue

    data = response.json()

    # Open DART 응답 코드가 000이면 정상입니다.
    if data.get("status") != "000":
        print(
            f'{company["input_name"]}: '
            f'{data.get("status")} - {data.get("message")}'
        )
        continue

    # 우리 프로젝트의 기업 ID도 함께 넣어 나중에 연결할 수 있게 합니다.
    data["company_id"] = company["company_id"]
    data["input_name"] = company["input_name"]
    results.append(data)

print(f"조회 성공: {len(results)}개 / 전체 {len(companies)}개")

조회 성공: 7개 / 전체 7개


In [10]:
result_df = pd.DataFrame(results)

# 요청한 대로 최대 10개까지만 표시합니다.
display(result_df.head(3))

,status,message,corp_code,corp_name,corp_name_eng,stock_name,stock_code,ceo_nm,corp_cls,jurir_no,...,adres,hm_url,ir_url,phn_no,fax_no,induty_code,est_dt,acc_mt,company_id,input_name
0,000,정상,00812353,(주)이니스프리,Innisfree Corporation,이니스프리,,최민정,E,1101114216522,...,서울특별시 용산구 한강대로 100 (한강로2가),www.innisfree.co.kr,,02-2186-7110,02-6040-7108,46443,20091110,12,COMPANY-INNISFREE,이니스프리
1,000,정상,01061150,주식회사 에스쁘아,ESPOIR CORPORATION,에스쁘아,,이연정,E,1101115605013,...,서울특별시 용산구 한강대로 100 (한강로2가),,,02-862-2604,02-6040-2700,46443,20150102,12,COMPANY-ESPOIR,에스쁘아
2,000,정상,00583424,(주)아모레퍼시픽,AMOREPACIFIC CORP.,아모레퍼시픽,090430,"서경배, 김승환",Y,1101113471036,...,서울특별시 용산구 한강대로 100 (한강로2가),www.apgroup.com,,02-6040-5114,02-6040-6349,20423,20060601,12,COMPANY-AMOREPACIFIC,아모레퍼시픽


In [16]:
pd.set_option("display.max_colwidth", None)

In [17]:
# 재무 관련 데이터 조회 

url = "https://opendart.fss.or.kr/api/list.json"
reports = []

for company in companies:
    page = 1

    while True:
        response = requests.get(
            url,
            params={
                "crtfc_key": dart_api_key,
                "corp_code": company["dart_corp_code"],
                "bgn_de": "20260101",
                "end_de": "20261104",
                "last_reprt_at": "Y",  # 최종보고서만 조회
                "page_count": 100,
                "page_no": page,
            },
            timeout=30,
        )

        if response.status_code != 200:
            print(f'{company["input_name"]}: HTTP 오류 {response.status_code}')
            break

        data = response.json()

        if data.get("status") != "000":
            print(
                f'{company["input_name"]}: '
                f'{data.get("status")} - {data.get("message")}'
            )
            break

        for report in data.get("list", []):
            title = report.get("report_nm", "")

            if "감사보고서" in title and "2025.12" in title:
                receipt_no = report["rcept_no"]

                reports.append({
                    "company_id": company["company_id"],
                    "기업명": company["input_name"],
                    "보고서명": title,
                    "접수일": report.get("rcept_dt"),
                    "접수번호": receipt_no,
                    "보고서 링크": (
                        f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={receipt_no}"
                    ),
                })

        if page >= int(data.get("total_page", 1)):
            break

        page += 1

reports_df = pd.DataFrame(reports)
display(reports_df.head(10))

,company_id,기업명,보고서명,접수일,접수번호,보고서 링크
0,COMPANY-INNISFREE,이니스프리,감사보고서 (2025.12),20260401,20260401003769,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260401003769
1,COMPANY-ESPOIR,에스쁘아,감사보고서 (2025.12),20260326,20260326001214,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260326001214
2,COMPANY-ETUDE,에뛰드,감사보고서 (2025.12),20260331,20260331004546,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260331004546
3,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,감사보고서 (2025.12),20260326,20260326000513,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260326000513
4,COMPANY-COSRX,코스알엑스,감사보고서 (2025.12),20260402,20260402000034,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260402000034
